In [ ]:
import s3fs
import rasterio
import xarray as xr
import rioxarray
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import boto3
import pandas as pd
import geopandas as gpd
import sqlite3

from functools import reduce

from const import HOST_SPCODES, COARSEN_FACTOR

## Check that treemap is in scratch bucket

In [ ]:
fs = s3fs.S3FileSystem()
fs.ls("s3://nasa-cryo-scratch/s-kganz/treemap/Data/")

## TreeMap database structure

In [ ]:
# It seems like downloading the database is the best option. Save to temp so we
# don't have extra data lying around.
fs.download(
    "nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016_tree_table.db",
    "/tmp/TreeMap2016_tree_table.db"
)

In [ ]:
db = duckdb.connect("/tmp/TreeMap2016_tree_table.db")
db.sql("SHOW TABLES")

In [ ]:
db.sql("PRAGMA table_info(TreeMap2016_tree_table)")

In [ ]:
db.sql("SELECT * FROM TreeMap2016_tree_table LIMIT 5").df()

## Summarize host BA per damage-causing agent and TM ID

In [ ]:
# pi/4 * DIA (in.) * DIA (in.) * TPA (1/ac) = BA (in^2/ac) 
# BA (in^2) * pi/4 (-) * 6.4516e-4 (m^2/in^2) * 247.105 (ac/km^2) = BA (m^2/km^2)
ba_conversion_factor = 6.4516e-4 * 247.105 * np.pi / 4
print(ba_conversion_factor)

In [ ]:
# Then iterate through each damage-causing agent, query, and assign values
# to TM IDs that have nonzero basal area.
ba_by_dca = []
for (dca, spcodes) in HOST_SPCODES.items():
    query = f'''
    SELECT 
        tm_id,
        SUM(DIA * DIA * TPA_UNADJ * {ba_conversion_factor}) as {dca}_ba
    FROM TreeMap2016_tree_table
    WHERE 
        STATUSCD == 1 AND
        SPCD IN {spcodes}
    GROUP BY tm_id
    HAVING {dca}_ba > 0
    '''
    #print(query)
    hostba_where_present = db.sql(query).df()
    hostba_where_present = hostba_where_present.set_index("tm_id")
    ba_by_dca.append(hostba_where_present)

## Population-weighted hydrologic traits per TM ID

In [ ]:
# Read the median trait file
traits = pd.read_excel("../data_in/tree_traits/GlobalTrees_Traits_Median.xlsx", sheet_name="GlobalTraits_Median")
traits = traits.set_index("spec.name")
traits.head()

In [ ]:
# Figure out what scientific names are in treemap and verify that these are all present
# in the traits database.
all_spcodes = reduce(set.union, map(set, HOST_SPCODES.values()))
print(f"Found {len(all_spcodes)} species")

all_spec_names = db.sql(f'''
    SELECT DISTINCT SCIENTIFIC_NAME
    FROM TreeMap2016_tree_table
    WHERE SPCD IN {tuple(all_spcodes)}
''').df()
all_spec_names = all_spec_names.set_index("SCIENTIFIC_NAME")

In [ ]:
all_spec_names.iloc[~all_spec_names.index.isin(traits.index)]

In [ ]:
# Recode spp that failed to match
all_spec_names = all_spec_names.rename({
    "Pinus washoensis": "Pinus ponderosa",
    "Abies lasiocarpa var. arizonica": "Abies lasiocarpa"
})

# Assert that everything matches now
assert all_spec_names.index.isin(traits.index).all()

In [ ]:
# Subset the traits table. Ignore height bc we get that from FIA
trait_cols = ["gsmax", "P50", "rdmax", "WUE"]
traits_subset = traits.loc[all_spec_names.index][trait_cols]

In [ ]:
# Insert this table into the sqlite database so duckdb can use it
con = sqlite3.connect("/tmp/TreeMap2016_tree_table.db")
traits_subset.to_sql("Treemap_Median_Traits", con)

In [ ]:
# Verify that it worked
db.sql("SHOW TABLES")

In [ ]:
# Calculate the tree-density-weighted average of height and traits. Note this excludes
# all the non beetle-host species.
hydro_summary = db.sql(f'''
    SELECT
        tm_id,
        SUM(gsmax * TPA_UNADJ) / SUM(TPA_UNADJ) AS gsmax,
        SUM(P50 * TPA_UNADJ) / SUM(TPA_UNADJ) AS P50,
        SUM(rdmax * TPA_UNADJ) / SUM(TPA_UNADJ) AS rdmax,
        SUM(WUE * TPA_UNADJ) / SUM(TPA_UNADJ) AS WUE,
        SUM(HT * TPA_UNADJ) / SUM(TPA_UNADJ) as HT
    FROM
    (
        TreeMap2016_tree_table
        INNER JOIN Treemap_Median_Traits
        ON TreeMap2016_tree_table.SCIENTIFIC_NAME = Treemap_Median_Traits.SCIENTIFIC_NAME
    )
    GROUP BY tm_id
''').df().set_index("tm_id")

## Concatenate BA tables and hydro trait table

Above tables only have TM IDs that contain beetle host species. For later, we need every TM ID. So make a series of all TM IDs but no columns.

In [ ]:
all_tmids = db.sql(
    """
    SELECT DISTINCT tm_id FROM TreeMap2016_tree_table
    """
).df().set_index("tm_id")
print(all_tmids.shape)

In [ ]:
host_hydro = pd.concat([all_tmids, hydro_summary] + ba_by_dca, axis=1)
# Fill only the BA columns with zero, hydro traits remain nan
ba_cols = filter(lambda x: x.endswith("_ba"), host_hydro.columns)
host_hydro = host_hydro.fillna({c: 0 for c in ba_cols})
host_hydro

In [ ]:
# Account for nodata pixels. We want zero BA (these are nonforest)
# and nan for hydro traits so they don't influence trait resampling.
nodata_tmid = 2147483647
host_hydro.loc[nodata_tmid] = ([np.nan] * 5) + ([0] * 8)
host_hydro.tail()

## Open treemap from scratch bucket

In [ ]:
session = rasterio.session.AWSSession(boto3.Session(), requester_pays=True)

# This defines how much memory we use in map_blocks.
# Must be a multiple of COARSEN_FACTOR.
chunks = dict(x=3000, y=3000)
treemap = rioxarray.open_rasterio(
    "s3://nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016.tif", 
    band_as_variable=True,
    chunks=chunks
)
print(treemap.rio.crs)
treemap

In [ ]:
# Figure out processing extent
usfs_regions = gpd.read_file("../data_in/usfs_region_boundaries/usfs_regions_simple.shp").to_crs(treemap.rio.crs)
usfs_regions_explode = usfs_regions.geometry.explode()
# Ignore Hawaii (sorry Hawaii)
usfs_regions_explode = usfs_regions_explode[usfs_regions_explode.geometry.area > 2e11]
bounds = usfs_regions_explode.total_bounds
xmin, ymin, xmax, ymax = bounds
print(bounds)

In [ ]:
# sel() must be exactly on the beginning/end of a chunk for ease of use
# with map_blocks(). So snap to the nearest coordinate and then snap
# to the nearest chunk.
x_snap = treemap.x.sel(x=[xmin, xmax], method="nearest")
y_snap = treemap.y.sel(y=[ymin, ymax], method="nearest")
x_idx = np.where(treemap.x.isin(x_snap))[0]
y_idx = np.where(treemap.y.isin(y_snap))[0]

print("Before snapping:", x_idx, y_idx)

x_idx[0] = int(chunks["x"] * (np.floor(x_idx[0] / chunks["x"])))
x_idx[1] = int(chunks["x"] * (np.ceil(x_idx[1] / chunks["x"])))
y_idx[0] = int(chunks["y"] * (np.floor(y_idx[0] / chunks["y"])))
y_idx[1] = int(chunks["y"] * (np.ceil(y_idx[1] / chunks["y"])))

print("After snapping:", x_idx, y_idx)

In [ ]:
treemap_clip = treemap.isel(
    x=slice(*x_idx),
    y=slice(*y_idx) 
)
treemap_clip

In [ ]:
# Assert that all chunks are the same size.
for dim in treemap_clip.chunksizes:
    sizes = np.array(treemap_clip.chunksizes[dim])
    assert (sizes == sizes[0]).all()

## Process using map_blocks()

The goal here is to replace each pixel in TreeMap with the corresponding entry in the above table. There are many more treemap IDs in the above table than unique values in each block, so it would be very efficient to np.where() for everything. Instead we use fast Pandas indexing with .loc[] to transfer the relevant rows of the table to the pixel. Then to fit everything in memory we coarsen to 3 km pixels.

The flatten step pulls all the data in the chunk into memory, so the best option for processing the full array is `xr.map_blocks`.

In [ ]:
# Make a local cluster for parallelism
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

#cluster = LocalCluster(n_workers=3, memory_limit="12GiB")
#client = cluster.get_client()
#client

In [ ]:
def process_block(block: xr.Dataset) -> xr.DataArray:
    block_tmids = block.band_1.data.flatten()
    block_ba = xr.DataArray(
        data=host_hydro.loc[block_tmids].to_numpy().reshape(block.band_1.shape + (host_hydro.shape[1],)),
        dims=block.band_1.dims + ("var",),
        coords=dict(var=host_hydro.columns, **block.coords)
    )
    block_coarse = block_ba.coarsen(dict(y=COARSEN_FACTOR, x=COARSEN_FACTOR), boundary="trim").mean()
    return block_coarse

In [ ]:
template = treemap_clip.band_1.coarsen(x=COARSEN_FACTOR, y=COARSEN_FACTOR, boundary="trim").mean()
template = template.expand_dims(var=host_hydro.columns, axis=-1)
template

In [ ]:
host_hydro_arr = xr.map_blocks(
    func=process_block,
    obj=treemap_clip,
    template=template
)

In [ ]:
with ProgressBar():
    host_hydro_arr = host_hydro_arr.compute()

In [ ]:
host_hydro_arr

In [ ]:
host_hydro_arr.to_zarr("../data_working/treemap2016_hostba_hydro.zarr")

In [ ]:
from cartopy import crs as ccrs
from cartopy import feature as cfeature

source_proj = ccrs.AlbersEqualArea(
    central_latitude=23,
    central_longitude=-96,
    standard_parallels=(29.5, 45.5)
)
target_proj = ccrs.Mercator()

In [ ]:
host_hydro.columns

In [ ]:
fig, axes = plt.subplots(nrows=5, ncols=3, figsize=(8, 12), subplot_kw=dict(projection=target_proj))
xmin, ymin, xmax, ymax = host_hydro_arr.rio.transform_bounds(3857)

for var, ax in zip(host_hydro.columns, axes.flat):
    host_hydro_arr.sel(var=var).plot(ax=ax, transform=source_proj, add_labels=False, xlim=[xmin, xmax], ylim=[ymin, ymax])
    ax.set_title(var)
    ax.coastlines()
    ax.add_feature(cfeature.STATES)
    
plt.show()